<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #0284c7; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Introducción a Polars y Estructuras de Datos 🐻‍❄️⚡
      </h1>
      <p style="margin: 6px 0 0 0; color: #0284c7; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Polars de Alto Rendimiento
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #0284c7; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Módulo 11 Extra
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #0284c7; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/11%20-%20Polars/00_Introduccion_Polars_y_Estructuras_de_Datos.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## 1. ¿Qué es Polars y Por Qué Revoluciona la Ciencia de Datos? 🚀

**Polars** es una biblioteca de procesamiento de datos de última generación diseñada desde cero para ofrecer **máxima velocidad y eficiencia de memoria**. A diferencia de Pandas, que nació en 2008 sobre NumPy y la memoria tradicional de Python, Polars fue creado implementando los avances más modernos de la ingeniería de software:

### Los Cuatro Pilares Arquitectónicos de Polars:
1. **Desarrollado en Rust:** Seguridad de memoria en tiempo de compilación y rendimiento nativo sin el bloqueo del GIL (*Global Interpreter Lock*).
2. **Basado en Apache Arrow:** Formato columnar contiguo en memoria que permite operaciones vectorizadas mediante instrucciones de CPU SIMD.
3. **Paralelismo Multihilo Automático:** Todos los núcleos del procesador trabajan de forma simultánea por defecto.
4. **Inmutabilidad y Cero Copias:** Facilita el paso de datos sin duplicar memoria y permite optimización previa de consultas.

### ¿Por Qué Importa Cada Pilar en la Práctica? 🔍

No basta con nombrar estos conceptos: entender su impacto real es lo que marca la diferencia al procesar datasets grandes.

| Pilar | Consecuencia Práctica |
|---|---|
| **Rust sin GIL** | Varios hilos pueden ejecutar código *de verdad* en paralelo (no simulado), algo que Python puro no permite por el candado global del intérprete. |
| **Apache Arrow columnar** | Al tener las columnas contiguas en memoria, el CPU aprovecha la caché y las instrucciones SIMD para procesar varios valores en un solo ciclo de reloj. |
| **Multihilo automático** | Una agregación sobre decenas de miles de filas se reparte sola entre todos los núcleos disponibles, sin que el usuario escriba `multiprocessing` ni configure nada. |
| **Cero copias / inmutabilidad** | Pasar un `DataFrame` a otra función (o exportarlo a Arrow/Pandas) no implica duplicar los datos en RAM, lo que reduce drásticamente el consumo de memoria en pipelines largos. |

> 💡 En resumen: Polars no es "Pandas pero más rápido por casualidad" — es una reingeniería completa que aprovecha hardware moderno (múltiples núcleos, SIMD, cachés de CPU) que Pandas, por su diseño heredado de los 2000, no puede explotar de la misma forma.

---
## 2. Comparativa: Pandas vs. Polars ⚖️

| Característica | Pandas 🐼 | Polars 🐻‍❄️ |
|---|---|---|
| **Lenguaje del Núcleo** | C, Cython y Python | **Rust puro** |
| **Estructura en Memoria** | Arreglos NumPy (bloques heterogéneos) | **Apache Arrow (columnar contiguo)** |
| **Paralelismo** | Monohilo por defecto | **Multihilo automático nativo** |
| **Consumo de Memoria** | Alto (5x a 10x del peso del archivo) | **Mínimo (aprox. 2x a 3x)** |
| **Modos de Ejecución** | Solo Eager (inmediato) | **Eager y Lazy (grafo optimizado)** |
| **Índice Explícito** | Posee índice (`df.index`) | **Sin índice (posicional natural)** |

In [17]:
import polars as pl
import numpy as np
import os

print(f"🚀 Polars versión: {pl.__version__}")

import os, urllib.request, urllib.parse

def load_dataset(filename, module_name="11 - Polars"):
    """
    Carga o descarga de forma segura el dataset para ejecución local o en Google Colab.
    Si no se encuentra localmente ni en GitHub, lo genera automáticamente.
    """
    candidates = [
        os.path.join("data", filename),
        os.path.join(module_name, "data", filename),
        os.path.join("..", "data", filename),
        os.path.join("..", module_name, "data", filename),
        os.path.join("Data Science programming", module_name, "data", filename),
        os.path.join("..", "Data Science programming", module_name, "data", filename),
        filename
    ]
    for path in candidates:
        if os.path.exists(path):
            return path
            
    os.makedirs("data", exist_ok=True)
    target_path = os.path.join("data", filename)
    folder_path = f"Data Science programming/{module_name}"
    encoded_folder = urllib.parse.quote(folder_path)
    encoded_file = urllib.parse.quote(filename)
    
    urls = [
        f"https://raw.githubusercontent.com/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/main/{encoded_folder}/data/{encoded_file}",
        f"https://raw.githubusercontent.com/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/master/{encoded_folder}/data/{encoded_file}"
    ]
    
    print(f"📥 Intentando descargar '{filename}' desde el repositorio oficial...")
    for url in urls:
        try:
            with urllib.request.urlopen(url, timeout=3) as response:
                if response.status == 200:
                    with open(target_path, 'wb') as out_f:
                        out_f.write(response.read())
                    print(f"✅ Dataset '{filename}' descargado exitosamente.")
                    return target_path
        except Exception:
            continue
            
    print(f"⚙️ Generando '{filename}' sintéticamente para ejecución inmediata...")
    import polars as pl
    import numpy as np
    np.random.seed(42)
    
    n_clientes = 1000
    df_c = pl.DataFrame({
        'id_cliente': [f'CLI-{i:04d}' for i in range(1, n_clientes + 1)],
        'nombre': [f'Cliente_{i}' for i in range(1, n_clientes + 1)],
        'segmento': np.random.choice(['Corporativo', 'Pyme', 'Consumo', 'Gobierno'], n_clientes),
        'edad': np.random.randint(18, 70, n_clientes),
        'ciudad_residencia': np.random.choice(['Tunja', 'Bogotá', 'Medellín', 'Cali', 'Bucaramanga'], n_clientes),
        'ingreso_anual': np.random.normal(45000000, 15000000, n_clientes).round(2)
    })
    
    n_ventas = 60000
    cats = ['Tecnología', 'Mobiliario', 'Material de Oficina', 'Servicios']
    prods = ['Laptop Pro', 'Monitor 4K', 'Silla Ergonómica', 'Escritorio', 'Papel A4', 'Tóner', 'Mantenimiento']
    ciudades = ['Tunja', 'Bogotá', 'Medellín', 'Cali', 'Barranquilla']
    
    cant = np.random.randint(1, 10, n_ventas)
    pu = np.random.choice([25000.0, 120000.0, 450000.0, 1200000.0, 3500000.0], n_ventas)
    desc = np.random.choice([0.0, 0.05, 0.10, 0.15], n_ventas)
    tot = (cant * pu * (1 - desc)).round(2)
    
    df_v = pl.DataFrame({
        'id_venta': [f'VNT-{i:06d}' for i in range(1, n_ventas + 1)],
        'fecha': [f'2024-{np.random.randint(1,13):02d}-{np.random.randint(1,29):02d}' for _ in range(n_ventas)],
        'id_cliente': np.random.choice(df_c['id_cliente'], n_ventas),
        'categoria': np.random.choice(cats, n_ventas),
        'producto': np.random.choice(prods, n_ventas),
        'cantidad': cant,
        'precio_unitario': pu,
        'descuento': desc,
        'ciudad_venta': np.random.choice(ciudades, n_ventas),
        'total_venta': tot
    })
    
    c_csv_path = os.path.join("data", "clientes.csv")
    c_pq_path = os.path.join("data", "clientes.parquet")
    v_csv_path = os.path.join("data", "ventas.csv")
    v_pq_path = os.path.join("data", "ventas.parquet")
    
    if not os.path.exists(c_csv_path): df_c.write_csv(c_csv_path)
    if not os.path.exists(c_pq_path): df_c.write_parquet(c_pq_path)
    if not os.path.exists(v_csv_path): df_v.write_csv(v_csv_path)
    if not os.path.exists(v_pq_path): df_v.write_parquet(v_pq_path)
    
    print(f"✅ Datasets preparados exitosamente en '{target_path}'.")
    return target_path

🚀 Polars versión: 1.35.2


---
## 3. Estructuras de Datos: `Series` y `DataFrame` 🧱

* **`pl.Series`:** Arreglo unidimensional homogéneo con tipo estricto.
* **`pl.DataFrame`:** Matriz bidimensional columnar compuesta por Series de igual longitud.

In [18]:
# 1. Serie
edades = pl.Series("edad", [23, 29, 34, 41, None, 27], dtype=pl.Int64)
print(edades)
print(f"Promedio: {edades.mean():.2f} | Nulos: {edades.null_count()}")

# 2. DataFrame
df_usuarios = pl.DataFrame({
    "id": [1, 2, 3, 4, 5],
    "nombre": ["Carlos", "Ana", "David", "Laura", "Felipe"],
    "ciudad": ["Tunja", "Bogotá", "Medellín", "Tunja", "Cali"],
    "salario": [3500000.0, 4200000.0, 5100000.0, 3800000.0, 4900000.0],
    "activo": [True, True, False, True, False]
})
display(df_usuarios)
print("Esquema:", df_usuarios.schema)

shape: (6,)
Series: 'edad' [i64]
[
	23
	29
	34
	41
	null
	27
]
Promedio: 30.80 | Nulos: 1


id,nombre,ciudad,salario,activo
i64,str,str,f64,bool
1,"""Carlos""","""Tunja""",3.5e6,true
2,"""Ana""","""Bogotá""",4.2e6,true
3,"""David""","""Medellín""",5.1e6,false
4,"""Laura""","""Tunja""",3.8e6,true
5,"""Felipe""","""Cali""",4.9e6,false


Esquema: Schema({'id': Int64, 'nombre': String, 'ciudad': String, 'salario': Float64, 'activo': Boolean})


---
## 4. Explorando un DataFrame en Profundidad: Atributos y Métodos Esenciales 🔬

Antes de transformar datos es indispensable saber "interrogarlos". Polars ofrece un conjunto rico de atributos y métodos para conocer la forma y la calidad de un `DataFrame` sin escribir bucles manuales:

| Atributo / Método | ¿Qué responde? |
|---|---|
| `.shape` | Tupla `(filas, columnas)`. |
| `.columns` | Lista con los nombres de las columnas. |
| `.schema` | Diccionario ordenado `{columna: dtype}` — el "documento de identidad" del DataFrame. |
| `.dtypes` | Lista de dtypes en el mismo orden que `.columns`. |
| `.describe()` | Estadísticas descriptivas (conteo, nulos, media, desviación, cuartiles, min/max) por columna. |
| `.null_count()` | Cantidad de valores nulos por columna, en un DataFrame de una sola fila. |
| `.n_unique()` (sobre una columna) | Número de valores distintos en esa Serie. |
| `.cast(dtype)` | Convierte explícitamente una columna a otro tipo de dato. |

Practiquemos todos estos sobre el `df_usuarios` que creamos arriba:

In [19]:
print("Forma (filas, columnas):", df_usuarios.shape)
print("Columnas:", df_usuarios.columns)
print("Esquema completo:", df_usuarios.schema)
print("Lista de dtypes:", df_usuarios.dtypes)

print("\nResumen estadístico:")
display(df_usuarios.describe())

print("\nConteo de nulos por columna:")
display(df_usuarios.null_count())

print(f"\nCiudades distintas: {df_usuarios['ciudad'].n_unique()}")

# Reducir el tipo de 'id' de Int64 a Int8 (ahorra memoria si los valores son pequeños)
df_usuarios_cast = df_usuarios.with_columns(pl.col("id").cast(pl.Int8))
print("\nEsquema tras el cast de 'id' a Int8:", df_usuarios_cast.schema)

Forma (filas, columnas): (5, 5)
Columnas: ['id', 'nombre', 'ciudad', 'salario', 'activo']
Esquema completo: Schema({'id': Int64, 'nombre': String, 'ciudad': String, 'salario': Float64, 'activo': Boolean})
Lista de dtypes: [Int64, String, String, Float64, Boolean]

Resumen estadístico:


statistic,id,nombre,ciudad,salario,activo
str,f64,str,str,f64,f64
"""count""",5.0,"""5""","""5""",5.0,5.0
"""null_count""",0.0,"""0""","""0""",0.0,0.0
"""mean""",3.0,null,null,4.3e6,0.6
"""std""",1.581139,null,null,689202.437605,null
"""min""",1.0,"""Ana""","""Bogotá""",3.5e6,0.0
"""25%""",2.0,null,null,3.8e6,null
"""50%""",3.0,null,null,4.2e6,null
"""75%""",4.0,null,null,4.9e6,null
"""max""",5.0,"""Laura""","""Tunja""",5.1e6,1.0



Conteo de nulos por columna:


id,nombre,ciudad,salario,activo
u32,u32,u32,u32,u32
0,0,0,0,0



Ciudades distintas: 4

Esquema tras el cast de 'id' a Int8: Schema({'id': Int8, 'nombre': String, 'ciudad': String, 'salario': Float64, 'activo': Boolean})


---
## 5. Tipado Estricto y Explícito: Detectando Errores Antes de que Cuesten Caro 🛡️

Una diferencia crucial —y poco publicitada— entre Pandas y Polars es **cómo manejan los tipos de datos (dtypes)**:

* **Pandas** es *permisivo*: si una columna numérica recibe un valor que no puede convertir, generalmente la reconvierte de forma silenciosa a tipo `object` (un "cajón de sastre" que mezcla enteros, strings y lo que sea). El problema queda oculto hasta que, mucho más adelante en el pipeline, una operación matemática falla de forma confusa.
* **Polars** es *estricto por diseño*: cada `Series` tiene un único dtype declarado y, si intentas construirla con valores incompatibles, **falla de inmediato** con un mensaje claro indicando qué valor causó el problema.

Esta filosofía se llama ***fail-fast*** (fallar rápido): es preferible detener el programa con un error explícito en el momento exacto del problema, que dejar pasar datos corruptos que exploten silenciosamente varias celdas (o varios días) después.

Comparemos el comportamiento de ambas librerías con un mismo dato "sucio": una lista de números que incluye por error un texto.

In [20]:
import pandas as pd

datos_sucios = [10, 20, "treinta", 40]

# 1. Comportamiento de Pandas: conversión silenciosa a 'object'
serie_pandas = pd.Series(datos_sucios)
print(f"Pandas -> dtype resultante: {serie_pandas.dtype} (sin avisar del problema)")
print(serie_pandas.tolist())

print("-" * 60)

# 2. Comportamiento de Polars: falla rápido (fail-fast) con dtype explícito
try:
    serie_polars = pl.Series("valores", datos_sucios, dtype=pl.Int64)
except Exception as error:
    print(f"Polars -> Error detectado de inmediato ({type(error).__name__}):")
    print(error)

print("-" * 60)

# 3. Si de verdad quieres permitir la mezcla, Polars te obliga a decidirlo
#    de forma explícita con strict=False, en vez de hacerlo por accidente
serie_flexible = pl.Series("valores", datos_sucios, dtype=pl.Int64, strict=False)
print("Polars (strict=False) -> convierte lo que puede y usa null en lo demás:")
print(serie_flexible)

Pandas -> dtype resultante: object (sin avisar del problema)
[10, 20, 'treinta', 40]
------------------------------------------------------------
Polars -> Error detectado de inmediato (TypeError):
unexpected value while building Series of type Int64; found value of type String: "treinta"

Hint: Try setting `strict=False` to allow passing data with mixed types.
------------------------------------------------------------
Polars (strict=False) -> convierte lo que puede y usa null en lo demás:
shape: (4,)
Series: 'valores' [i64]
[
	10
	20
	null
	40
]


---
## 6. Lectura Rápida de Archivos: CSV y Parquet 📂⚡

Polars puede leer archivos **CSV** y **Parquet** con una API casi idéntica (`pl.read_csv()` / `pl.read_parquet()`), pero el rendimiento y la fidelidad de tipos que ofrecen son muy distintos:

* **CSV** es texto plano: rápido de generar e inspeccionar a simple vista, pero Polars debe **inferir** el tipo de cada columna leyendo los datos, ya que el archivo no guarda ninguna información de tipos.
* **Parquet** es un formato **binario columnar** (el mismo paradigma que Apache Arrow): guarda los datos ya comprimidos, ya tipados y organizados por columna, por lo que Polars no necesita adivinar nada ni descomprimir texto — simplemente lee los bytes correctos.

Carguemos el dataset `ventas.csv` (60,000 filas de ventas comerciales) y démosle un primer vistazo con `.glimpse()`, un método pensado justamente para inspeccionar rápidamente forma y tipos:

In [21]:
ruta_csv = load_dataset("ventas.csv")
df_ventas = pl.read_csv(ruta_csv)
print(f"Dimensiones de ventas: {df_ventas.shape}")
df_ventas.glimpse()

📥 Intentando descargar 'ventas.csv' desde el repositorio oficial...
⚙️ Generando 'ventas.csv' sintéticamente para ejecución inmediata...
✅ Datasets preparados exitosamente en 'data/ventas.csv'.
Dimensiones de ventas: (60000, 10)
Rows: 60000
Columns: 10
$ id_venta        <str> 'VNT-000001', 'VNT-000002', 'VNT-000003', 'VNT-000004', 'VNT-000005', 'VNT-000006', 'VNT-000007', 'VNT-000008', 'VNT-000009', 'VNT-000010'
$ fecha           <str> '2024-11-09', '2024-04-10', '2024-05-27', '2024-10-15', '2024-09-12', '2024-02-23', '2024-06-19', '2024-05-25', '2024-06-23', '2024-01-27'
$ id_cliente      <str> 'CLI-0117', 'CLI-0775', 'CLI-0705', 'CLI-0592', 'CLI-0173', 'CLI-0644', 'CLI-0979', 'CLI-0718', 'CLI-0776', 'CLI-0994'
$ categoria       <str> 'Mobiliario', 'Mobiliario', 'Mobiliario', 'Mobiliario', 'Servicios', 'Material de Oficina', 'Servicios', 'Material de Oficina', 'Material de Oficina', 'Tecnología'
$ producto        <str> 'Laptop Pro', 'Monitor 4K', 'Laptop Pro', 'Laptop Pro', 'Laptop Pr

---
## 7. CSV vs. Parquet en la Práctica: Velocidad, Peso y Tipos ⚖️

Repitamos la carga, esta vez con el archivo `ventas.parquet` (el mismo dataset, en formato Parquet), y comparemos tres cosas: **tiempo de lectura**, **peso en disco** y **tipos de datos inferidos**.

In [22]:
import time

ruta_parquet = load_dataset("ventas.parquet")

# Tiempo de lectura CSV
t0 = time.time()
_ = pl.read_csv(ruta_csv)
t_csv = time.time() - t0

# Tiempo de lectura Parquet
t0 = time.time()
df_ventas_pq = pl.read_parquet(ruta_parquet)
t_parquet = time.time() - t0

print(f"Tiempo lectura CSV:     {t_csv * 1000:8.2f} ms")
print(f"Tiempo lectura Parquet: {t_parquet * 1000:8.2f} ms")

print(f"\nPeso en disco CSV:     {os.path.getsize(ruta_csv) / 1_048_576:6.2f} MB")
print(f"Peso en disco Parquet: {os.path.getsize(ruta_parquet) / 1_048_576:6.2f} MB")

print("\nEsquema inferido desde CSV:")
print(df_ventas.schema)
print("\nEsquema leído desde Parquet (tipos ya definidos en el archivo):")
print(df_ventas_pq.schema)

Tiempo lectura CSV:        61.29 ms
Tiempo lectura Parquet:    61.78 ms

Peso en disco CSV:       5.13 MB
Peso en disco Parquet:   0.43 MB

Esquema inferido desde CSV:
Schema({'id_venta': String, 'fecha': String, 'id_cliente': String, 'categoria': String, 'producto': String, 'cantidad': Int64, 'precio_unitario': Float64, 'descuento': Float64, 'ciudad_venta': String, 'total_venta': Float64})

Esquema leído desde Parquet (tipos ya definidos en el archivo):
Schema({'id_venta': String, 'fecha': String, 'id_cliente': String, 'categoria': String, 'producto': String, 'cantidad': Int64, 'precio_unitario': Float64, 'descuento': Float64, 'ciudad_venta': String, 'total_venta': Float64})


> 💡 **Qué esperar:** en datasets de este tamaño, Parquet suele leerse varias veces más rápido que CSV y ocupar una fracción de su peso en disco gracias a la compresión columnar. Además, fíjate en el `schema`: Parquet tiende a preservar tipos más ricos y compactos (por ejemplo, fechas como `Datetime` en vez de texto plano), mientras que CSV —al no llevar metadatos de tipo— obliga a Polars a inferirlos y suele quedarse con tipos más genéricos.

---
## 8. Segundo Dataset: `clientes.csv` y `clientes.parquet` 👥

El módulo también incluye un segundo dataset, `clientes` (1,500 filas), con información demográfica: `id_cliente`, `nombre`, `segmento`, `edad`, `ciudad_residencia` e `ingreso_anual`. Lo usaremos en la práctica guiada y en el ejercicio final.

In [23]:
ruta_clientes_csv = load_dataset("clientes.csv")
ruta_clientes_parquet = load_dataset("clientes.parquet")

df_clientes = pl.read_csv(ruta_clientes_csv)
df_clientes_pq = pl.read_parquet(ruta_clientes_parquet)

print(f"Filas x columnas (CSV):     {df_clientes.shape}")
print(f"Filas x columnas (Parquet): {df_clientes_pq.shape}")
df_clientes.glimpse()

Filas x columnas (CSV):     (1000, 6)
Filas x columnas (Parquet): (1000, 6)
Rows: 1000
Columns: 6
$ id_cliente        <str> 'CLI-0001', 'CLI-0002', 'CLI-0003', 'CLI-0004', 'CLI-0005', 'CLI-0006', 'CLI-0007', 'CLI-0008', 'CLI-0009', 'CLI-0010'
$ nombre            <str> 'Cliente_1', 'Cliente_2', 'Cliente_3', 'Cliente_4', 'Cliente_5', 'Cliente_6', 'Cliente_7', 'Cliente_8', 'Cliente_9', 'Cliente_10'
$ segmento          <str> 'Consumo', 'Gobierno', 'Corporativo', 'Consumo', 'Consumo', 'Gobierno', 'Corporativo', 'Corporativo', 'Consumo', 'Pyme'
$ edad              <i64> 34, 26, 50, 37, 30, 45, 65, 46, 30, 63
$ ciudad_residencia <str> 'Bogotá', 'Medellín', 'Tunja', 'Medellín', 'Medellín', 'Cali', 'Bucaramanga', 'Tunja', 'Medellín', 'Tunja'
$ ingreso_anual     <f64> 56607491.15, 61832625.43, 54513617.15, 52129283.84, 48472691.09, -5274157.75, 40686134.71, 30876157.24, 63161093.08, 54285557.97



---
## 9. Práctica Guiada: Resumen Estadístico y Conversión 🛠️

In [24]:
# Resumen estadístico descriptivo
display(df_ventas.describe())

# Conversión sin costo a Pandas
pdf = df_ventas.head(10).to_pandas()
print("Tipo exportado a pandas:", type(pdf))

statistic,id_venta,fecha,id_cliente,categoria,producto,cantidad,precio_unitario,descuento,ciudad_venta,total_venta
str,str,str,str,str,str,f64,f64,f64,str,f64
"""count""","""60000""","""60000""","""60000""","""60000""","""60000""",60000.0,60000.0,60000.0,"""60000""",60000.0
"""null_count""","""0""","""0""","""0""","""0""","""0""",0.0,0.0,0.0,"""0""",0.0
"""mean""",null,null,null,null,null,5.012883,1.0648e6,0.075146,null,4.9464e6
"""std""",null,null,null,null,null,2.575091,1.2943e6,0.056045,null,7.2481e6
"""min""","""VNT-000001""","""2024-01-01""","""CLI-0001""","""Material de Oficina""","""Escritorio""",1.0,25000.0,0.0,"""Barranquilla""",21250.0
"""25%""",null,null,null,null,null,3.0,120000.0,0.0,null,324000.0
"""50%""",null,null,null,null,null,5.0,450000.0,0.05,null,1.71e6
"""75%""",null,null,null,null,null,7.0,1.2e6,0.15,null,6.48e6
"""max""","""VNT-060000""","""2024-12-28""","""CLI-1000""","""Tecnología""","""Tóner""",9.0,3.5e6,0.15,"""Tunja""",3.15e7


Tipo exportado a pandas: <class 'pandas.core.frame.DataFrame'>


---
## 10. Ejercicio Práctico: Analiza a los Clientes 🧪

Usando el DataFrame `df_clientes` que cargamos en la Sección 8, resuelve lo siguiente:

1. Calcula el **ingreso anual promedio** agrupado por `segmento`, ordenado de mayor a menor. *(Pista: `group_by()` + `agg()` + `sort()` — los verás en detalle en el Cuaderno 02, pero puedes intentarlo con lo aprendido hasta aquí).*
2. Convierte (`.cast()`) la columna `edad` a `pl.Int32`.
3. Cuenta cuántos valores nulos existen en total en `df_clientes` con `.null_count()`.

Escribe tu solución en la celda de abajo antes de revisar la respuesta guiada:

In [25]:
# 1. Ingreso promedio por segmento, ordenado descendente
# ingreso_por_segmento = ...

# 2. Cast de 'edad' a Int32
# clientes_tipado = ...

# 3. Conteo de nulos
# nulos_totales = ...


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
# 1. Ingreso promedio por segmento, ordenado descendente
ingreso_por_segmento = (
    df_clientes
    .group_by("segmento")
    .agg(pl.col("ingreso_anual").mean().alias("ingreso_promedio"))
    .sort("ingreso_promedio", descending=True)
)
print(ingreso_por_segmento)

# 2. Cast de 'edad' a Int32
clientes_tipado = df_clientes.with_columns(pl.col("edad").cast(pl.Int32))
print(clientes_tipado.schema["edad"])

# 3. Conteo de nulos
nulos_totales = df_clientes.null_count()
print(nulos_totales)
```
</details>

---
## 11. Resumen y Próximos Pasos 📌

| Concepto | Idea Clave |
|---|---|
| **Arquitectura** | Rust + Apache Arrow + multihilo + cero copias = velocidad sin sacrificar seguridad de memoria. |
| **Tipado** | Polars es estricto y *fail-fast*: los errores de tipo se detectan al construir la Serie, no varias celdas después. |
| **`pl.Series`** | Arreglo 1D homogéneo con dtype único y explícito. |
| **`pl.DataFrame`** | Colección de Series del mismo largo; sin índice explícito. |
| **Exploración** | `.shape`, `.columns`, `.schema`, `.describe()`, `.null_count()`, `.n_unique()` — la caja de herramientas para "interrogar" un DataFrame. |
| **CSV vs. Parquet** | CSV es texto plano e inferido; Parquet es binario, tipado y más liviano/rápido de leer. |

➡️ **Siguiente paso:** en el **Cuaderno 01 — Expresiones, Contextos y Transformaciones** aprenderás el lenguaje de expresiones de Polars (`select`, `with_columns`, `filter`, `when/then/otherwise`) para transformar estos mismos datasets de forma declarativa y paralela.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Módulo Extra: Polars de Alto Rendimiento</i>
  </p>
</div>